# +90Gündem V22 — Evidence AI Newsroom
Cloudflare kuyruğu + çoklu kaynak kanıtı + AI editör + deterministik yayın kapısı.

Secrets: `OPENROUTER_API_KEY`, `AI_NEWSROOM_TOKEN`, `WORKER_URL`

In [ ]:
!pip -q install requests rapidfuzz
import json,requests,time
from urllib.parse import urlparse
from rapidfuzz.fuzz import token_set_ratio
from google.colab import userdata
OPENROUTER_API_KEY=userdata.get('OPENROUTER_API_KEY');AI_NEWSROOM_TOKEN=userdata.get('AI_NEWSROOM_TOKEN');WORKER_URL=userdata.get('WORKER_URL').rstrip('/')
MODEL='openai/gpt-4.1-mini';H={'Authorization':f'Bearer {AI_NEWSROOM_TOKEN}'}
assert OPENROUTER_API_KEY and AI_NEWSROOM_TOKEN and WORKER_URL
print('✅ V22 bağlantıları hazır')

In [ ]:
TRUST={'aa.com.tr':95,'trthaber.com':90,'resmigazete.gov.tr':100,'kap.org.tr':100,'tff.org':98,'uefa.com':98,'fifa.com':98,'reuters.com':98,'apnews.com':98,'bbc.com':94,'ntv.com.tr':86,'haberturk.com':84}
def dom(u):
 try:return urlparse(u or '').netloc.lower().removeprefix('www.')
 except:return ''
def trust(u):return TRUST.get(dom(u),55 if dom(u) else 35)
def cluster(primary,candidates,threshold=67):
 base=(primary.get('title','')+' '+primary.get('description','')).strip();out=[]
 for x in candidates:
  sim=token_set_ratio(base,(x.get('title','')+' '+x.get('description','')).strip())
  if sim>=threshold and x.get('id')!=primary.get('id'):out.append({**x,'similarity':sim,'source_trust':trust(x.get('link'))})
 return sorted(out,key=lambda x:(x['source_trust'],x['similarity']),reverse=True)
def evidence(primary,candidates):
 m=cluster(primary,candidates);allx=[primary]+m;ds={dom(x.get('link')) for x in allx if dom(x.get('link'))};official=any(trust(x.get('link'))>=98 for x in allx);avg=sum(trust(x.get('link')) for x in allx)/len(allx)
 score=min(100,round(35+min(30,max(0,len(ds)-1)*15)+min(20,avg*.20)+(15 if official else 0)))
 return {'evidence_score':score,'independent_source_count':len(ds),'official_or_wire_confirmed':official,'corroborating_sources':m[:5]}
print('✅ Kanıt motoru hazır')

In [ ]:
SYSTEM='''Sen +90Gündem kıdemli Türkçe haber editörüsün. Sana primary haber ve deterministik evidence verilir. Kanıtı yok sayma. Bilgi uydurma. Tek kaynaklı iddialarda temkinli ol. JSON alanları: title,summary,category,entities,ai_verification_score,viral_score,image_query,x_text,reasons. Skorlar 0-100. x_text kısa, doğal, en fazla 2 alakalı hashtag.'''
def ai_edit(bundle):
 r=requests.post('https://openrouter.ai/api/v1/chat/completions',headers={'Authorization':f'Bearer {OPENROUTER_API_KEY}','Content-Type':'application/json'},json={'model':MODEL,'temperature':0.1,'response_format':{'type':'json_object'},'messages':[{'role':'system','content':SYSTEM},{'role':'user','content':json.dumps(bundle,ensure_ascii=False)}]},timeout=60);r.raise_for_status();return json.loads(r.json()['choices'][0]['message']['content'])
def fuse(result,ev):
 ai=max(0,min(100,int(result.get('ai_verification_score',0))));e=ev['evidence_score'];n=ev['independent_source_count'];official=ev['official_or_wire_confirmed']
 final=round(e*.65+ai*.35)
 # Tek kaynağın AI tarafından şişirilmesini engelle
 if n<2 and not official:final=min(final,69)
 if n<2 and official:final=min(90,max(final,75))
 result['evidence_score']=e;result['verification_score']=final;result['independent_source_count']=n;result['official_or_wire_confirmed']=official
 viral=int(result.get('viral_score',0))
 if final<55:decision='reject'
 elif final<75 or viral<45:decision='review'
 else:decision='publish'
 result['publish_decision']=decision;return result
print('✅ Evidence + AI füzyon kapısı hazır')

In [ ]:
def pull(limit=10):
 r=requests.get(WORKER_URL+'/ai-newsroom/pull',params={'limit':limit},headers=H,timeout=30);r.raise_for_status();return r.json().get('items',[])
def post(path,data):
 r=requests.post(WORKER_URL+path,json=data,headers=H,timeout=30);r.raise_for_status();return r.json()
def process_once(limit=10):
 items=pull(limit);out=[]
 for item in items:
  post('/ai-newsroom/claim',{'id':item['id']})
  try:
   ev=evidence(item,items);bundle={'primary':item,**ev};result=fuse(ai_edit(bundle),ev)
  except Exception as e:result={'publish_decision':'review','verification_score':0,'error':str(e)[:300]}
  post('/ai-newsroom/result',{'id':item['id'],'result':result});out.append({'id':item['id'],'decision':result['publish_decision'],'evidence':result.get('evidence_score'),'verification':result.get('verification_score'),'viral':result.get('viral_score')})
 return out
print('✅ İşleme hattı hazır')

In [ ]:
# TEK HÜCRE ÇALIŞTIRMA
results=process_once(10);print(json.dumps(results,ensure_ascii=False,indent=2))

### Yayın güvenliği
`verification_score = %65 deterministik evidence + %35 AI`. Bağımsız ikinci kaynak yoksa ve resmî/ajans teyidi yoksa skor **69'u geçemez**, dolayısıyla otomatik publish olamaz. Resmî/birincil kaynak tek başına mevcutsa kontrollü biçimde publish eşiğine ulaşabilir. AI doğrudan X'e göndermez.